# 01 — Data Exploration

Explore the processed Adult Income dataset and its attached SHAP values.
This notebook is for exploration only — do not run the pipeline from here.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

load_dotenv('../.env')
sns.set_theme(style='whitegrid', context='notebook')

In [ ]:
from src.config import load_config
from src.data_loader import load_dataset, get_shap_columns, get_feature_columns, format_shap_table

cfg = load_config('../config/default.yaml')
print('Datasets:', [d.name for d in cfg.datasets])
print('Models:  ', [m.id for m in cfg.models])

## Adult Income dataset

In [ ]:
adult_cfg = cfg.get_dataset('adult')
adult = load_dataset(adult_cfg)
print(f'Shape: {adult.shape}')
adult.head()

In [ ]:
shap_cols = get_shap_columns(adult, adult_cfg.shap_col_prefix)
feat_cols = get_feature_columns(adult, adult_cfg.shap_col_prefix)
print('Feature columns:', feat_cols)
print('SHAP columns:   ', shap_cols)

In [ ]:
# Distribution of SHAP values across all instances
fig, axes = plt.subplots(len(shap_cols), 1, figsize=(10, 2 * len(shap_cols)))
if len(shap_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, shap_cols):
    adult[col].hist(bins=40, ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(col)
    ax.set_xlabel('SHAP value')
plt.tight_layout()
plt.show()

In [ ]:
# Mean absolute SHAP value per feature (global importance)
mean_abs = adult[shap_cols].abs().mean().sort_values(ascending=False)
mean_abs.index = [c.replace(adult_cfg.shap_col_prefix, '') for c in mean_abs.index]

fig, ax = plt.subplots(figsize=(8, 4))
mean_abs.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_ylabel('Mean |SHAP|')
ax.set_title('Global feature importance — Adult Income')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Inspect a single instance prompt
row = adult.iloc[0]
print(format_shap_table(row, adult_cfg.shap_col_prefix))